### Enter full names of group members:

##### Name A: Markus Aleksander Råkil Johansen
##### Name B: Frederik Andreas Brunvoll Farstad

In [127]:
import math
import numpy as np
from sympy import prime
from pathlib import Path  # for paths of files
import csv
import copy
import random
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from hashlib import md5

# ANSI escape codes for colors
class colors:
    red = '\033[91m'
    green = '\033[92m'
    blue = '\033[94m'
    end = '\033[0m'  


### 1. DGIM

#### 1.1. DGIM algorithm

In [128]:
# Default DGIM parameters

stream_path = 'data/my_stream.txt'

# The window size
N = 500 

In [129]:
class Bucket:
    def __init__(self, size:int, timestamp):
        self.size = size
        self.timestamp = timestamp

    def __repr__(self):
        return f"Bucket(size={self.size}, timestamp={self.timestamp})"
    
    def get_timestamp(self):
        return self.timestamp
    
    def get_size(self) -> int:
        return self.size
    


def count_sizes(bucket_list: list[Bucket]):
    """
    Count the number of buckets of each size in the bucket list.
    Args:
        bucket_list (list): List of Bucket objects.
    Returns:
        dict: A dictionary where keys are bucket sizes and values are lists of buckets of that size.
    """
    size_counts = {}
    for bucket in bucket_list:
        size = bucket.get_size()
        if size not in size_counts:
            size_counts[size] = []
        size_counts[size].append(bucket)
    return size_counts



def merge_buckets(bucket_list, bucket1:Bucket, bucket2:Bucket):
    """
    Merge two buckets into one by summing their sizes and keeping the timestamp of the older bucket.
    This function modifies the original bucket list in place.
    It removes the two buckets being merged and adds a new bucket with the combined size.
    """
    bucket_list.remove(bucket1)
    bucket_list.remove(bucket2)
    bucket_list.append(Bucket(
        size=bucket1.get_size() + bucket2.get_size(),
        timestamp=bucket1.get_timestamp()
    ))



def dgim_algorithm(stream_path, N):
    
    # Create the buckets list and initialize the timestamp
    bucket_list = []
    timestamp = 0

    # Loop through the entire data stream, one bit at a time
    with open(stream_path) as f:
        
        while True:
            bit = f.read(1)
            if not bit:
                break
            
            # Update the timestamp and remove buckets older than N (outside the current window)
            timestamp += 1 #Update the timestamp, which is the number of bits read so far.
            if len(bucket_list) > 0:
                bucket_list = [bucket for bucket in bucket_list if bucket.get_timestamp() > timestamp - N]
            
            # If the current bit is 1, create a new bucket
            if bit == '1':
                # Create a new bucket as a dictionary
                bucket_list.append(Bucket(1, timestamp))
                
                # Merge buckets if necessary
                need_to_merge = True
                while need_to_merge:
                    need_to_merge = False
                    
                    # Count the sizes of the buckets
                    size_counts = count_sizes(bucket_list)
                    
                    # Check for sizes that need merging (more than 2 buckets of same size)
                    for size in sorted(size_counts.keys()):
                        if len(size_counts[size]) > 2:
                            need_to_merge = True
                            
                            # Sort by timestamp (oldest first)
                            size_counts[size].sort(key=lambda bucket: bucket.get_timestamp())
                
                            # Merge the oldest two buckets of this size
                            merge_buckets(bucket_list, size_counts[size][0], size_counts[size][1])
                            
                            # Break to recalculate sizes
                            break
    
    #Return the final bucket list and the timestamp
    return bucket_list, timestamp

In [130]:
bucket = dgim_algorithm(stream_path, N)

In [131]:
print(f"The updated list of timestamps buckets from DGIM algorithm: \n {bucket[0]}")
print(f"The end timestamp: {bucket[1]}")   

The updated list of timestamps buckets from DGIM algorithm: 
 [Bucket(size=64, timestamp=1009690), Bucket(size=8, timestamp=1010045), Bucket(size=16, timestamp=1010007), Bucket(size=32, timestamp=1009951), Bucket(size=64, timestamp=1009823), Bucket(size=4, timestamp=1010076), Bucket(size=8, timestamp=1010064), Bucket(size=2, timestamp=1010090), Bucket(size=4, timestamp=1010084), Bucket(size=1, timestamp=1010099), Bucket(size=2, timestamp=1010094)]
The end timestamp: 1010102


#### 1.2. Query the Bucket 

In [132]:
def actual_count(stream_path, k):
    stream_list = []
    with open(stream_path, 'r') as file:
        for line in file:
            stream_list.extend(list(map(int, line.strip())))

    # Convert the list into a numpy array
    stream_array = np.array(stream_list)
    
    return int(np.sum(stream_array[-k:]))

In [133]:
def dgim_query(bucket, N, k):      
    # Extract the buckets and the end timestamp
    bucket_list, end_time_stamp = bucket
   
    # Initialize the different variables
    one_count = 0
    
    # Validate query range
    if k > N:
        raise ValueError("Query range k cannot exceed window size N")
    
    # Calculate the start timestamp for our query window
    query_start = end_time_stamp - k + 1
    
    # Sort buckets by timestamps (newest first)
    sorted_buckets = sorted(bucket_list, key=lambda bucket: bucket.get_timestamp(), reverse=True)
    
    # Process the buckets
    for bucket in sorted_buckets:
        if bucket.get_timestamp() >= query_start:
            # Bucket is entirely within our query window
            one_count += bucket.get_size()
        else:
            # This bucket crosses the boundary (partially in window)
            # We estimate that half of its bits are in our window
            one_count += bucket.get_size() / 2
            break
    
    return math.ceil(one_count)

In [134]:
# List of queries
K = [10, 50, 100, 300, 500] 

In [135]:
print("---------------------------------------------------------------")
for k in K:
    dgim_count = dgim_query(bucket, 500, k)
    true_count = actual_count(stream_path, k)
    
    print(f"The total 1s in the last {k} bits by DGIM: {dgim_count}")
    print(f"The true count of 1s in the last {k} bits: {true_count}")
    print(f"The DGIM error for predicted 1s in the last {k} bits: \
    {round(abs(100*(dgim_count-true_count))/true_count,2)} %")
    print("---------------------------------------------------------------")

---------------------------------------------------------------
The total 1s in the last 10 bits by DGIM: 4
The true count of 1s in the last 10 bits: 5
The DGIM error for predicted 1s in the last 10 bits:     20.0 %
---------------------------------------------------------------
The total 1s in the last 50 bits by DGIM: 25
The true count of 1s in the last 50 bits: 26
The DGIM error for predicted 1s in the last 50 bits:     3.85 %
---------------------------------------------------------------
The total 1s in the last 100 bits by DGIM: 61
The true count of 1s in the last 100 bits: 51
The DGIM error for predicted 1s in the last 100 bits:     19.61 %
---------------------------------------------------------------
The total 1s in the last 300 bits by DGIM: 173
The true count of 1s in the last 300 bits: 150
The DGIM error for predicted 1s in the last 300 bits:     15.33 %
---------------------------------------------------------------
The total 1s in the last 500 bits by DGIM: 205
The true 

### 2. Bloom filters

In [136]:
# Username data for the creation of bloom filters - B
data_file = (Path("data/bloom_username").with_suffix('.csv'))

# Test data to check the functionality and false positive rate
test1_file = (Path("data/test1_username").with_suffix('.csv'))
test2_file = (Path("data/test2_username").with_suffix('.csv'))

# Default bloom filter parameters
bloom_size = 1500000 # parameter N
h = 3 # number of hash functions

In [137]:
# create an array of bloom filter with zeros
B = np.zeros(bloom_size)

In [138]:
B

array([0., 0., 0., ..., 0., 0., 0.], shape=(1500000,))

#### 2.1. Create Bloom filter

In [139]:
def generate_hash(h, N):
    hash_list = []

    # To-do! generate a list of hash functions
    for i in range(h):
        # Create different hash functions using different random seeds
        def hash_func(x, seed=i):
            # Convert hexdigest to integer before applying modulo
            return int(md5((str(x) + str(seed)).encode("utf-8")).hexdigest(), 16) % N
        hash_list.append(hash_func)

    return hash_list

In [140]:
hashes = generate_hash(h, bloom_size)

In [141]:
def create_bloom_filter(B, hashes, data):
    with data.open() as f:
        for name in f:
            
            # To-do! update the hash index of the bloom filter with 1s
            for hash_func in hashes:
                index = hash_func(name)
                B[index] = 1
            
    return B

In [142]:
bloom_array = create_bloom_filter(B, hashes, data_file)

In [143]:
bloom_array

array([0., 0., 1., ..., 0., 1., 1.], shape=(1500000,))

#### 2.2. Verify usernames

In [144]:
def single_verify_username(bloom_array, hashes, new_user):
    
    # To-do! verify username and return a code of 0 or 1 (1 - username taken and 0 - username available)
    # Check if all of the hashes of the new username are 1 in the bloom array
    for hash_func in hashes:
        index = hash_func(new_user)
        if bloom_array[index] == 0:
            return 0
    return 1
    

In [145]:
# Feel free to test different usernames here

new_username = "KazeemTDT4305"

#new_username = "ShambaTDT4305"

In [146]:
user_code = single_verify_username(bloom_array, hashes, new_username)

In [147]:
if user_code == 1:
    print(colors.red + f"Username {new_username} has been taken. Try again!" + colors.end)
elif user_code == 0:
    print(colors.green + f"Username {new_username} is available. Congrats!" + colors.end)
else:
    print(colors.blue + f"Wrong pass code. Please reverify!" + colors.end)  

Username KazeemTDT4305 is available. Congrats!


In [148]:
def group_verify_username(bloom_array, hashes, data):
    # Initialize counts
    total_name = 0
    taken_name = 0
    
    with data.open() as f:
        for name in f:
            # To-do! similar to the single verify, but returns a percentage of usernames taken...
            # ...(In other words seen already by the bloom filter during its creation)
            if single_verify_username(bloom_array, hashes, name) == 1:
                taken_name += 1
            total_name += 1
            
    return round(taken_name/total_name*100,2)

In [149]:
print("----------------------------------------------------------")
user_total = group_verify_username(bloom_array, hashes, test1_file)
print(f"Percentage of username seen before from test 1: {user_total}%")
print("----------------------------------------------------------")
user_total = group_verify_username(bloom_array, hashes, test2_file)
print(f"Percentage of username seen before from test 2: {user_total}%")
print("----------------------------------------------------------")

----------------------------------------------------------
Percentage of username seen before from test 1: 100.0%
----------------------------------------------------------
Percentage of username seen before from test 2: 23.87%
----------------------------------------------------------


### 3. Flajolet-Martin

In [150]:
def flajolet_martin(input_stream):
    R = 0  # Initialize maximum rightmost zero bit position to 0

    # To-do! Define hash function h(x) = 6x + 1 mod 5

    hash_function = lambda x: (6 * x + 1) % 5

    # To-do! Iterate over the input stream and update maximum rightmost zero bit position
    for bit in input_stream:
        # hash the bit
        hash_value = hash_function(bit)
        # convert to binary and remove the '0b' prefix
        binary_hash = bin(hash_value)[2:]
        # count the number of trailing zeros
        trailing_zeros = len(binary_hash) - len(binary_hash.rstrip("0"))

        if trailing_zeros > R:
            R = trailing_zeros

    # Estimate the number of distinct elements
    distinct_estimate = 2 ** R

    return distinct_estimate

In [151]:
# Input stream
input_stream1 = [1, 1, 2, 1, 2, 1, 1, 1, 1, 2, 1, 1]
input_stream2 = [1, 3, 2, 1, 2, 3, 4, 3, 1, 2, 3, 1]

# Run the Flajolet-Martin algorithm
distinct_estimate1 = flajolet_martin(input_stream1)
distinct_estimate2 = flajolet_martin(input_stream2)

# Print the estimated number of distinct elements
print("-----------------------------------------------------")
print(f"Distinct elements (estimated) in input stream 1:", distinct_estimate1)
print("-----------------------------------------------------")
print(f"Distinct elements (estimated) in input stream 2:", distinct_estimate2)
print("-----------------------------------------------------")

-----------------------------------------------------
Distinct elements (estimated) in input stream 1: 2
-----------------------------------------------------
Distinct elements (estimated) in input stream 2: 4
-----------------------------------------------------


### 4. Adword 

#### 4.1. Greedy Algorithm

In [152]:
# User queries
queries = ["big data", "big data", "big data","bloom filters", "bloom filters", "bloom filters",
           "flajolet martin", "flajolet martin", "flajolet martin", "dgim algorithm", "dgim algorithm", "dgim algorithm"]

In [153]:
# Company A B C and D keywords and budget $$$
global_companies = {
        'A': ["big data", "bloom filters", 3],
        'B': ["flajolet martin", 3],
        'C': ["flajolet martin", "dgim algorithm", 3],
        'D': ["big data", 3],
    }

In [154]:
def greedy_algorithm(local_companies, queries):
    # Initial revenue
    revenue = 0

    # To-do! update revenue using greedy algorithm

    for query in queries:
        candidate_companies = []
        for company in local_companies:
            if query in local_companies[company]:
                if local_companies[company][-1] > 0:
                    candidate_companies.append(company)

        if len(candidate_companies) > 0:
            # Choose random company
            company = random.choice(candidate_companies)
            revenue += 1
            local_companies[company][-1] -= 1
            #print(query, company)

    return revenue

In [155]:
total_revenue = 0
total_trials = 10
print("Starting trials using Greedy Algorithm...")
print("------------------------------------------------")
for i in range(total_trials):
    local_companies = copy.deepcopy(global_companies)
    revenue = greedy_algorithm(local_companies, queries)
    total_revenue = total_revenue + revenue
    print(f"Trial {i+1} - Revenue generated: {revenue}")
print("------------------------------------------------")   
print("Average revenue generated for all trials: ",total_revenue/total_trials)

Starting trials using Greedy Algorithm...
------------------------------------------------
Trial 1 - Revenue generated: 8
Trial 2 - Revenue generated: 9
Trial 3 - Revenue generated: 9
Trial 4 - Revenue generated: 8
Trial 5 - Revenue generated: 11
Trial 6 - Revenue generated: 7
Trial 7 - Revenue generated: 8
Trial 8 - Revenue generated: 10
Trial 9 - Revenue generated: 10
Trial 10 - Revenue generated: 8
------------------------------------------------
Average revenue generated for all trials:  8.8


#### 4.2. Balance Algorithm

In [156]:
def balance_algorithm(local_companies, queries):
    # Initial revenue
    revenue = 0

    # To-do! update revenue using balance algorithm

    for query in queries:
        candidate_companies = []
        for company in local_companies:
            if query in local_companies[company]:
                if local_companies[company][-1] > 0:
                    candidate_companies.append(company)

        if len(candidate_companies) > 0:
            # Companies with highest budget in a list
            highest_budget_companies = [company for company in candidate_companies if local_companies[company][-1] == max(local_companies[company][-1] for company in candidate_companies)]

            # Choose random company from the list
            company = random.choice(highest_budget_companies)
            
            revenue += 1
            local_companies[company][-1] -= 1
            #print(query, company)
            

    return revenue

In [157]:
total_revenue = 0
total_trials = 10
print("Starting trials using Balance Algorithm...")
print("-------------------------------------------")
for i in range(total_trials):
    local_companies = copy.deepcopy(global_companies)
    revenue = balance_algorithm(local_companies, queries)
    total_revenue = total_revenue + revenue
    print(f"Trial {i+1} - Revenue generated: {revenue}")
print("-------------------------------------------")   
print("Average revenue generated for all trials: ",total_revenue/total_trials)

Starting trials using Balance Algorithm...
-------------------------------------------
Trial 1 - Revenue generated: 9
Trial 2 - Revenue generated: 9
Trial 3 - Revenue generated: 9
Trial 4 - Revenue generated: 9
Trial 5 - Revenue generated: 10
Trial 6 - Revenue generated: 8
Trial 7 - Revenue generated: 9
Trial 8 - Revenue generated: 10
Trial 9 - Revenue generated: 9
Trial 10 - Revenue generated: 9
-------------------------------------------
Average revenue generated for all trials:  9.1


### 5. Recommender System

In [158]:
# Ratings matrix (each row corresponds to a movie, and each column corresponds to a user)
ratings_matrix = np.array([
    [1, 0, 3, 0, 0, 5, 0, 0, 5, 0, 4, 0],
    [0, 0, 5, 4, 0, 0, 4, 0, 0, 2, 1, 3],
    [2, 4, 0, 1, 2, 0, 3, 0, 4, 3, 5, 0],
    [0, 2, 4, 0, 5, 0, 0, 4, 0, 0, 2, 0],
    [0, 0, 4, 3, 4, 2, 0, 0, 0, 0, 2, 5],
    [1, 0, 3, 0, 3, 0, 0, 2, 0, 0, 4, 0]
])

#### 5.1. User-User Collaborative Filtering

In [159]:
def user_cf(rate_m, tup_mu, neigh):
    # Create a deep copy of the ratings matrix
    #print(rate_m)

    # Transpose the ratings matrix to get users as rows and movies as columns
    ratings_matrix = copy.deepcopy(rate_m).astype(float).T

    #print(ratings_matrix)

    # Get the target movie and user indices
    target_movie, target_user = tup_mu
    

    # Calculate the cosine similarity between all users
    similarity_matrix = cosine_similarity(ratings_matrix)

    #print(pd.DataFrame(similarity_matrix))

    # Get the indices of the most similar users
    similar_users = np.argsort(similarity_matrix[target_user-1])[::-1][1:neigh+1]

    #print(similar_users)

    # Get the similarity values for the most similar users
    similar_user_similarities = similarity_matrix[target_user - 1, similar_users]

    # Calculate the weighted average of ratings
    # If all similarities are zero, use a simple average
    if np.sum(similar_user_similarities) == 0:
        predicted_rating = np.mean(rate_m[target_movie - 1, similar_users])
    else:
        # Weight the ratings by similarity
        weighted_ratings = (
            rate_m[target_movie - 1, similar_users] * similar_user_similarities
        )
        predicted_rating = np.sum(weighted_ratings) / np.sum(similar_user_similarities)

    return predicted_rating

In [160]:
# List of tuple of movie rating by users to be predicted e.g (1, 5) refers to the rating of movie 1 by user 5
list_mu_query = [(1, 5), (3, 3)]

# Neighbor selection (|N|)
neigh = 2

In [161]:
print("-----------------------------------------------------------------")   
for mu_query in list_mu_query:
    predicted_rating = user_cf(ratings_matrix, mu_query, neigh)
    print(f"The predicted rating of movie {mu_query[0]} by user {mu_query[1]}: {predicted_rating:.2f} (User-User CF)")
    print("-----------------------------------------------------------------")   

-----------------------------------------------------------------
The predicted rating of movie 1 by user 5: 1.42 (User-User CF)
-----------------------------------------------------------------
The predicted rating of movie 3 by user 3: 1.49 (User-User CF)
-----------------------------------------------------------------


#### 5.2. Item-Item Collaborative Filtering

In [165]:
def item_cf(rate_m, tup_mu, neigh):
    # Create a deep copy of the ratings matrix in order to avoid modifying the original matrix
    ratings_matrix = copy.deepcopy(rate_m).astype(float)
    
    target_movie, target_user = tup_mu  # Target movie and user indices

    
    # Calculate the cosine similarity between all items in the ratings matrix
    similarity_matrix = cosine_similarity(ratings_matrix.T)
    
    # Find indices of the most similar items to the target movie.
    similar_items = np.argsort(similarity_matrix[target_movie-1])[::-1][1:neigh+1] # Get the indices of the most similar items
    similar_item_similarities = similarity_matrix[target_movie-1, similar_items] # Get the similarity values for the most similar items
    
    # Get the user's ratings for similar items
    user_ratings = ratings_matrix[target_user-1, similar_items] # Get the user's ratings for similar items
    
    # Calculate the weighted average of ratings
    # If all similarities are zero, use a simple average
    if np.sum(similar_item_similarities) == 0:
        predicted_rating = np.mean(user_ratings)
    else:
        # Weight the ratings by similarity
        weighted_ratings = user_ratings * similar_item_similarities
        predicted_rating = np.sum(weighted_ratings) / np.sum(similar_item_similarities)
    
    return predicted_rating

In [166]:
print("-----------------------------------------------------------------")   
for mu_query in list_mu_query:
    predicted_rating = item_cf(ratings_matrix, mu_query, neigh)
    print(f"The predicted rating of movie {mu_query[0]} by user {mu_query[1]}: {predicted_rating:.2f} (Item-Item CF)")
    print("-----------------------------------------------------------------")   

-----------------------------------------------------------------
The predicted rating of movie 1 by user 5: 1.04 (Item-Item CF)
-----------------------------------------------------------------
The predicted rating of movie 3 by user 3: 1.49 (Item-Item CF)
-----------------------------------------------------------------


### Provide concise answers to all 5 cases in the Project 3 description below

#### Case 1

The DGIM algorithm stores O(log N) bits for the timestamp in each of the O(log N) buckets, which results in a space complexity of
O(log N * log N) = O(log^2 N). 

#### Case 2
- Your favorite social media platform uses Bloom filters to assign new usernames. When you try to obtain a new username, ’Kazeem,’ it says, "username is taken." Is there a possibility that the username is available? If yes, how do you present your case to the site admin?
- After a bit of ingenuity on your part, you finally came up with ’KazeemTDT4305", and fortunately, it says this is available. However, a friend sitting next to you prompted that they know a friend of a friend that has this exact username on the same platform. Is this possible? Knowing that this friend has never taken a CS course on Big Data and has probably always doubted your brilliance. Clearly, this is the moment you have been waiting for to brandish your TDT4305 Big Data prowess. How can you effectively convince your friend that he is mistaken?

1. The username may still be available. Since all the usernames are hashed into the same bit string, it is possible that multiple hashed usernames together fill all the bits corresponding to the "Kazeem" username, making it look like it is taken even though it is not.

2. The username has not been taken. The bloom filter effectively stores data about all the usernames that have been taken. If any of the hashed values return a 0 in the bloom filter, the username is definetively available. If the username is then made, the bloom filter will return a 1 for all the hashed values going forwards. If all the hashed values return a 1, the username is probably taken, but it may also be available (see previous answer).

#### Case 3

- How do we increase precision while using the Flajolet-Martin algorithm?

To increase the precision of the Flajolet-Martin algorithm, we can increase the number of hash functions used. This will reduce the probability of false positives and false negatives.

In practice, we "group" the hash functions together. In each group, we take the median of the number of trailing zeros. This effectively "smooths out" the outliers. Between the groups, we take the average of the medians. Using this average to calculate the number of distinct elements gives us a more accurate estimate.


#### Case 4

- What is the minimum and maximum possible revenue, and the competitive ratio of both the
Greedy and Balance algorithm for the input data provided in the Notebook?

### Greedy
- Minimum revenue: 6
- Maximum revenue: 12
- Competitive ratio: 1/2
### Balance
- Minimum revenue: 8
- Maximum revenue: 12
- Competitive ratio: 2/3

#### Case 5

In [164]:
# Enter answer here